# 도's Pick! · 3번

대회 시간: **2026-10-09 08:50~10:20 (한국 시간)**. 시작 전에는 운영자 개방 승인이 필요합니다.

MAICON 모의고사 1회차 | 문제 3
군용 장비 운영 데이터 기반 고장 여부 예측

[문제 설명]
군용 정비 시설에서는 장비의 온도, 회전 속도, 토크 및 누적 마모 정보를 기록하여 고장을 조기에 발견하려 합니다. 주어진 운영 데이터를 분석하고, 각 장비 관측값에서 고장이 발생했는지 여부를 예측하는 이진 분류 모델을 구축하세요.

[제공 데이터]
- train.csv: 장비 종류(Type), 공기·공정 온도, 회전 속도, 토크, 마모 시간 및 고장 여부
- test.csv: 고장 여부 정답이 제거된 테스트 관측값
- sample_submission.csv: 제출 예시

[예측 대상]
- machine_failure = 0: 정상
- machine_failure = 1: 고장

[모델 및 구현 조건]
1. RandomForestClassifier를 기본 모델로 학습하고 성능을 평가하세요. 다른 모델을 이용한 성능 개선도 허용됩니다.
2. 범주형 변수 인코딩, 파생 변수 생성, 클래스 불균형 대응, 검증 및 임곗값 최적화 등을 자유롭게 활용할 수 있습니다.
3. 원본 데이터의 Machine failure 외에도 TWF, HDF, PWF, OSF, RNF는 고장 발생 여부 또는 유형을 알려주는 정답 관련 열입니다. 이 다섯 열은 학습 및 테스트 입력 특성으로 사용할 수 없습니다.

[평가]
고장(1)을 양성 클래스로 하는 F1-score를 사용합니다. 고장 사례는 정상 사례보다 매우 적으므로 단순 정확도보다 양성 클래스의 precision과 recall을 함께 고려하세요. 높을수록 좋습니다.

[제출 형식]
파일명: submission.csv
컬럼: id,machine_failure
- machine_failure는 0 또는 1의 정수여야 합니다.
- 테스트 관측값마다 정확히 한 행을 작성하고, sample_submission.csv의 id와 행 순서를 유지하세요.

[원본 데이터]
Predictive Maintenance Dataset (AI4I 2020)
https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020

[태그]
#표형데이터 #전처리 #불균형분류 #고장예측 #RandomForest

## 이번 대회의 제공 구성
- 공개 학습: `data/train.csv` 8,000행. 정상 7,729행, 고장 271행
- 최종 평가: `data/test.csv` 2,000행. 정답은 공개하지 않습니다.
- 고장 여부를 층화해 80:20으로 분할했습니다. 학습·평가 양쪽에 두 클래스가 있습니다.
- `Product ID`, `TWF`, `HDF`, `PWF`, `OSF`, `RNF`는 입력에서 제거했습니다.

| 컬럼 | 의미 |
|---|---|
| id | 원본 UDI 기준 고유 식별자 |
| type | 장비 종류: L/M/H |
| air_temperature_k | 공기 온도(K) |
| process_temperature_k | 공정 온도(K) |
| rotational_speed_rpm | 회전 속도(rpm) |
| torque_nm | 토크(Nm) |
| tool_wear_min | 누적 마모 시간(min) |
| machine_failure | 학습에만 있는 정답. 0 정상 / 1 고장 |

## 공통 제출 검증
- 최종 파일: `/workspace/problem03/submission.csv`
- 정확한 컬럼 순서: `id,machine_failure`
- sample_submission의 모든 ID를 정확히 한 번씩, 같은 순서로 제출합니다. 누락·추가·중복·순서 변경은 오류입니다.
- UTF-8 또는 UTF-8 BOM CSV, 쉼표 구분, 헤더 포함. 빈 필드·NaN·무한대·깨진 CSV는 오류입니다.
- 1번 수량은 0 이상의 정수, 2번은 지정된 5개 문자열, 3번은 문자열 `0` 또는 `1` 형태의 정수 클래스입니다.
- 유효하지 않은 제출은 해당 문제 0점·ERROR로 처리합니다.
- 문제 점수는 **100 × raw 평가 지표**입니다. 정답과 정확히 같으면 100점이며 세 문제 합계는 300점입니다.
- 제공 데이터는 읽기 전용이며, 가공 결과·모델·제출은 작업공간에 저장합니다.


In [ ]:
!pip install xgboost==3.4.1 catboost


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
DATA_DIR = Path("data")
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
print("train/test/sample:", train.shape, test.shape, sample.shape)
display(train.head())


train/test/sample: (8000, 8) (2000, 7) (2000, 2)
   id type  air_temperature_k  process_temperature_k  rotational_speed_rpm  torque_nm  tool_wear_min  machine_failure
0   1    M              298.1                  308.6                  1551       42.8              0                0
1   3    L              298.1                  308.5                  1498       49.4              5                0
2   4    L              298.2                  308.6                  1433       39.5              7                0
3   5    L              298.2                  308.7                  1408       40.0              9                0
4   7    L              298.1                  308.6                  1558       42.4             14                0


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

# Explicit allowlist excludes identifiers and all failure-type target columns.
num = ['air_temperature_k', 'process_temperature_k', 'rotational_speed_rpm',
       'torque_nm', 'tool_wear_min']
features = num + ['type']
prep = ColumnTransformer([
    ('numeric', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num),
    ('category', OneHotEncoder(handle_unknown='ignore'), ['type']),
])
# Standardization is fitted inside the pipeline, using training rows only.
model = make_pipeline(prep, RandomForestClassifier(
    n_estimators=300, n_jobs=2, random_state=42))
X, y = train[features], train.machine_failure
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
model.fit(Xa, ya)
pred = model.predict(Xb)
metrics = {'f1': f1_score(yb, pred, zero_division=0),
           'precision': precision_score(yb, pred, zero_division=0),
           'recall': recall_score(yb, pred, zero_division=0)}
print('Holdout rows:', len(yb), 'positive:', int(yb.sum()))
print('Threshold: 0.5; metrics:', metrics)
print('Confusion matrix [normal, failure]:')
print(confusion_matrix(yb, pred, labels=[0, 1]))
scaler = model[0].named_transformers_['numeric'][1]
assert scaler.n_samples_seen_ == len(Xa)
model.fit(X, y)
submission = pd.DataFrame({'id': test.id,
    'machine_failure': model.predict(test[features]).astype(int)})
submission = sample[['id']].merge(submission, on='id', how='left', validate='one_to_one')
assert submission.machine_failure.isin([0, 1]).all()
submission.to_csv('submission.csv', index=False)
print('Test prediction counts:', submission.machine_failure.value_counts().to_dict())


Holdout rows: 1600 positive: 54
Threshold: 0.5; metrics: {'f1': 0.5853658536585366, 'precision': 0.8571428571428571, 'recall': 0.4444444444444444}
Confusion matrix [normal, failure]:
[[1542    4]
 [  30   24]]
Test prediction counts: {0: 1954, 1: 46}


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier

# Same features, scaler, split and decision threshold as the RF baseline.
gb_model = make_pipeline(clone(prep), GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=3,
    min_samples_leaf=5, random_state=42))
gb_model.fit(Xa, ya)
gb_pred = gb_model.predict(Xb)
gb_metrics = {'f1': f1_score(yb, gb_pred, zero_division=0),
              'precision': precision_score(yb, gb_pred, zero_division=0),
              'recall': recall_score(yb, gb_pred, zero_division=0)}
comparison = pd.DataFrame([metrics, gb_metrics], index=['RF300', 'GB200'])
print('Same stratified holdout, threshold=0.5:')
print(comparison.to_string())
print('GB confusion matrix [normal, failure]:')
print(confusion_matrix(yb, gb_pred, labels=[0, 1]))


Same stratified holdout, threshold=0.5:
             f1  precision    recall
RF300  0.585366   0.857143  0.444444
GB200  0.666667   0.733333  0.611111
GB confusion matrix [normal, failure]:
[[1534   12]
 [  21   33]]


In [ ]:
from xgboost import XGBClassifier
from sklearn.base import clone

# Refit RF on the training partition: preceding cells may have fitted it on all rows.
rf_eval = clone(model).fit(Xa, ya)
xgb_model = make_pipeline(clone(prep), XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=1,
    reg_lambda=1.0, objective='binary:logistic', eval_metric='logloss',
    tree_method='hist', n_jobs=2, random_state=42))
xgb_model.fit(Xa, ya)
valid_prob = np.column_stack([
    rf_eval.predict_proba(Xb)[:, 1],
    gb_model.predict_proba(Xb)[:, 1],
    xgb_model.predict_proba(Xb)[:, 1]])
rows = []
for name, prob in zip(['RF', 'GB', 'XGB', 'SoftVoting'],
                      [*valid_prob.T, valid_prob.mean(axis=1)]):
    pred = (prob >= 0.5).astype(int)
    rows.append({'model': name, 'F1': f1_score(yb, pred, zero_division=0),
                 'precision': precision_score(yb, pred, zero_division=0),
                 'recall': recall_score(yb, pred, zero_division=0)})
ensemble_comparison = pd.DataFrame(rows).set_index('model')
print(ensemble_comparison.to_string())
print('Ensemble confusion matrix:')
print(confusion_matrix(yb, (valid_prob.mean(axis=1) >= 0.5).astype(int)))

# Fixed equal weights; all preprocessing is refitted on full training data.
full_models = [clone(m).fit(X, y) for m in [rf_eval, gb_model, xgb_model]]
test_probability = np.mean([m.predict_proba(test[features])[:, 1]
                            for m in full_models], axis=0)
submission = pd.DataFrame({'id': test.id,
    'machine_failure': (test_probability >= 0.5).astype(int)})
submission = sample[['id']].merge(submission, on='id', how='left', validate='one_to_one')
assert submission.id.tolist() == sample.id.tolist()
assert submission.machine_failure.isin([0, 1]).all()
submission.to_csv('submission.csv', index=False)
print('Ensemble submission counts:', submission.machine_failure.value_counts().to_dict())


                  F1  precision    recall
model                                    
RF          0.585366   0.857143  0.444444
GB          0.666667   0.733333  0.611111
XGB         0.610526   0.707317  0.537037
SoftVoting  0.673913   0.815789  0.574074
Ensemble confusion matrix:
[[1539    7]
 [  23   31]]
Ensemble submission counts: {0: 1951, 1: 49}


In [ ]:
from catboost import CatBoostClassifier

# Same standardized numeric features and one-hot categories as RF/XGBoost.
cat_model = make_pipeline(clone(prep), CatBoostClassifier(
    iterations=300, depth=6, learning_rate=0.05,
    loss_function='Logloss', random_seed=42, thread_count=2,
    verbose=False, allow_writing_files=False))
cat_model.fit(Xa, ya)
new_valid_prob = np.column_stack([rf_eval.predict_proba(Xb)[:, 1],
    xgb_model.predict_proba(Xb)[:, 1], cat_model.predict_proba(Xb)[:, 1]])
new_rows = []
for name, prob in [('CatBoost', new_valid_prob[:, 2]),
                   ('RF+XGB+CatBoost', new_valid_prob.mean(axis=1)),
                   ('RF+GB+XGB', valid_prob.mean(axis=1))]:
    label = (prob >= 0.5).astype(int)
    new_rows.append({'model': name, 'F1': f1_score(yb, label, zero_division=0),
        'precision': precision_score(yb, label, zero_division=0),
        'recall': recall_score(yb, label, zero_division=0)})
cat_comparison = pd.DataFrame(new_rows).set_index('model')
print(cat_comparison.to_string())
print('RF+XGB+CatBoost confusion matrix:')
print(confusion_matrix(yb, (new_valid_prob.mean(axis=1) >= 0.5).astype(int)))
cat_full_models = [clone(m).fit(X, y) for m in [rf_eval, xgb_model, cat_model]]
cat_test_prob = np.mean([m.predict_proba(test[features])[:, 1]
                        for m in cat_full_models], axis=0)
submission = pd.DataFrame({'id': test.id,
    'machine_failure': (cat_test_prob >= 0.5).astype(int)})
submission = sample[['id']].merge(submission, on='id', how='left', validate='one_to_one')
assert submission.id.tolist() == sample.id.tolist()
assert submission.machine_failure.isin([0, 1]).all()
submission.to_csv('submission.csv', index=False)
print('Prediction counts:', submission.machine_failure.value_counts().to_dict())


                       F1  precision    recall
model                                         
CatBoost         0.666667   0.833333  0.555556
RF+XGB+CatBoost  0.659341   0.810811  0.555556
RF+GB+XGB        0.673913   0.815789  0.574074
RF+XGB+CatBoost confusion matrix:
[[1539    7]
 [  24   30]]
Prediction counts: {0: 1950, 1: 50}


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sample = pd.read_csv('data/sample_submission.csv')
base_num = ['air_temperature_k', 'process_temperature_k',
            'rotational_speed_rpm', 'torque_nm', 'tool_wear_min']
added_num = ['temperature_gap_k', 'mechanical_power_w', 'torque_wear']
def sensor_features(data):
    x = data[base_num + ['type']].copy()
    x['temperature_gap_k'] = x.process_temperature_k - x.air_temperature_k
    x['mechanical_power_w'] = x.torque_nm * x.rotational_speed_rpm * (2 * np.pi / 60)
    x['torque_wear'] = x.torque_nm * x.tool_wear_min
    return x

def sensor_models(numeric):
    def pipeline(estimator):
        prep = ColumnTransformer([
            ('numeric', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
            ('category', OneHotEncoder(handle_unknown='ignore'), ['type'])])
        return make_pipeline(prep, estimator)
    return [pipeline(RandomForestClassifier(n_estimators=300, n_jobs=2, random_state=42)),
        pipeline(XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=1, reg_lambda=1,
            objective='binary:logistic', eval_metric='logloss', tree_method='hist',
            n_jobs=2, random_state=42)),
        pipeline(CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05,
            loss_function='Logloss', random_seed=42, thread_count=2,
            verbose=False, allow_writing_files=False))]

sensor_X = sensor_features(train)
sensor_test = sensor_features(test)
y = train.machine_failure
oof = {name: np.zeros(len(train)) for name in ['original', 'sensor']}
fold_results = []
for fold, (a, b) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(sensor_X, y), 1):
    result = {'fold': fold}
    for name, numeric in [('original', base_num), ('sensor', base_num + added_num)]:
        fitted = sensor_models(numeric)
        for model in fitted:
            model.fit(sensor_X.iloc[a], y.iloc[a])
        oof[name][b] = np.mean([m.predict_proba(sensor_X.iloc[b])[:, 1] for m in fitted], axis=0)
        result[name] = f1_score(y.iloc[b], oof[name][b] >= 0.5)
    fold_results.append(result)
    print(result, flush=True)
comparison = pd.DataFrame([{'features': name,
    'F1': f1_score(y, prob >= 0.5),
    'precision': precision_score(y, prob >= 0.5),
    'recall': recall_score(y, prob >= 0.5)} for name, prob in oof.items()]).set_index('features')
print('Pooled 5-fold OOF, fixed threshold 0.5:')
print(comparison.to_string())
# The validation is used for model selection; it is not an untouched final test.
use_sensor = comparison.loc['sensor', 'F1'] > comparison.loc['original', 'F1']
selected_numeric = base_num + added_num if use_sensor else base_num
final_sensor_models = sensor_models(selected_numeric)
for model in final_sensor_models:
    model.fit(sensor_X, y)
sensor_probability = np.mean([m.predict_proba(sensor_test)[:, 1] for m in final_sensor_models], axis=0)
submission = pd.DataFrame({'id': test.id, 'machine_failure': (sensor_probability >= 0.5).astype(int)})
submission = sample[['id']].merge(submission, on='id', how='left', validate='one_to_one')
assert submission.id.tolist() == sample.id.tolist()
assert submission.machine_failure.isin([0, 1]).all()
submission.to_csv('submission.csv', index=False)
print('Selected sensor features:', use_sensor)
print('Submission counts:', submission.machine_failure.value_counts().to_dict())


{'fold': 1, 'original': 0.723404255319149, 'sensor': 0.8461538461538461}
{'fold': 2, 'original': 0.6744186046511628, 'sensor': 0.8686868686868687}
{'fold': 3, 'original': 0.6666666666666666, 'sensor': 0.8333333333333334}
{'fold': 4, 'original': 0.7777777777777778, 'sensor': 0.875}
{'fold': 5, 'original': 0.7032967032967034, 'sensor': 0.8932038834951457}
Pooled 5-fold OOF, fixed threshold 0.5:
                F1  precision    recall
features                               
original  0.709534   0.888889  0.590406
sensor    0.863454   0.947137  0.793358
Selected sensor features: True
Submission counts: {0: 1940, 1: 60}


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sample = pd.read_csv('data/sample_submission.csv')
base_num = ['air_temperature_k', 'process_temperature_k',
            'rotational_speed_rpm', 'torque_nm', 'tool_wear_min']
added_num = ['temperature_gap_k', 'mechanical_power_w', 'torque_wear']
def sensor_features(data):
    x = data[base_num + ['type']].copy()
    x['temperature_gap_k'] = x.process_temperature_k - x.air_temperature_k
    x['mechanical_power_w'] = x.torque_nm * x.rotational_speed_rpm * (2 * np.pi / 60)
    x['torque_wear'] = x.torque_nm * x.tool_wear_min
    return x

def sensor_models(numeric):
    def pipeline(estimator):
        prep = ColumnTransformer([
            ('numeric', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
            ('category', OneHotEncoder(handle_unknown='ignore'), ['type'])])
        return make_pipeline(prep, estimator)
    return [pipeline(RandomForestClassifier(n_estimators=300, n_jobs=2, random_state=42)),
        pipeline(XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=1, reg_lambda=1,
            objective='binary:logistic', eval_metric='logloss', tree_method='hist',
            n_jobs=2, random_state=42)),
        pipeline(CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05,
            loss_function='Logloss', random_seed=42, thread_count=2,
            verbose=False, allow_writing_files=False))]

sensor_X = sensor_features(train)
sensor_test = sensor_features(test)
y = train.machine_failure
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve

# Select threshold strictly within outer training data; outer labels are evaluation only.
threshold_oof = np.zeros(len(train))
threshold_labels = np.zeros(len(train), dtype=int)
threshold_records = []
for fold, (outer_a, outer_b) in enumerate(StratifiedKFold(
        n_splits=5, shuffle=True, random_state=42).split(sensor_X, y), 1):
    inner_a, inner_b = train_test_split(outer_a, test_size=0.25,
        stratify=y.iloc[outer_a], random_state=100 + fold)
    inner_models = sensor_models(base_num + added_num)
    for estimator in inner_models:
        estimator.fit(sensor_X.iloc[inner_a], y.iloc[inner_a])
    inner_prob = np.mean([m.predict_proba(sensor_X.iloc[inner_b])[:, 1]
                          for m in inner_models], axis=0)
    precision, recall, thresholds = precision_recall_curve(y.iloc[inner_b], inner_prob)
    f_scores = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    # For equal F1 choose the highest threshold (fewer false alarms).
    selected = float(thresholds[np.flatnonzero(np.isclose(f_scores, f_scores.max()))[-1]])
    outer_models = sensor_models(base_num + added_num)
    for estimator in outer_models:
        estimator.fit(sensor_X.iloc[outer_a], y.iloc[outer_a])
    prob = np.mean([m.predict_proba(sensor_X.iloc[outer_b])[:, 1]
                    for m in outer_models], axis=0)
    threshold_oof[outer_b] = prob
    threshold_labels[outer_b] = (prob >= selected).astype(int)
    row = {'fold': fold, 'threshold': selected,
           'F1_at_0.5': f1_score(y.iloc[outer_b], prob >= 0.5),
           'F1_tuned': f1_score(y.iloc[outer_b], prob >= selected)}
    threshold_records.append(row)
    print(row, flush=True)
threshold_comparison = pd.DataFrame([{'method': name,
    'F1': f1_score(y, labels), 'precision': precision_score(y, labels),
    'recall': recall_score(y, labels)} for name, labels in
    [('fixed_0.5', threshold_oof >= 0.5), ('inner_selected', threshold_labels)]]).set_index('method')
print('Outer OOF evaluation:')
print(threshold_comparison.to_string())
print('Evaluation only; submission retained.')


{'fold': 1, 'threshold': 0.3736157040658789, 'F1_at_0.5': 0.8461538461538461, 'F1_tuned': 0.8518518518518519}
{'fold': 2, 'threshold': 0.4920973734848713, 'F1_at_0.5': 0.8686868686868687, 'F1_tuned': 0.8686868686868687}
{'fold': 3, 'threshold': 0.5271505136636153, 'F1_at_0.5': 0.8333333333333334, 'F1_tuned': 0.8333333333333334}
{'fold': 4, 'threshold': 0.36070421029895944, 'F1_at_0.5': 0.875, 'F1_tuned': 0.8686868686868687}
{'fold': 5, 'threshold': 0.6515748779555418, 'F1_at_0.5': 0.8932038834951457, 'F1_tuned': 0.9}
Outer OOF evaluation:
                      F1  precision    recall
method                                       
fixed_0.5       0.863454   0.947137  0.793358
inner_selected  0.864542   0.939394  0.800738
Evaluation only; submission retained.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sample = pd.read_csv('data/sample_submission.csv')
base_num = ['air_temperature_k', 'process_temperature_k',
            'rotational_speed_rpm', 'torque_nm', 'tool_wear_min']
added_num = ['temperature_gap_k', 'mechanical_power_w', 'torque_wear']
def sensor_features(data):
    x = data[base_num + ['type']].copy()
    x['temperature_gap_k'] = x.process_temperature_k - x.air_temperature_k
    x['mechanical_power_w'] = x.torque_nm * x.rotational_speed_rpm * (2 * np.pi / 60)
    x['torque_wear'] = x.torque_nm * x.tool_wear_min
    return x

def sensor_models(numeric):
    def pipeline(estimator):
        prep = ColumnTransformer([
            ('numeric', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
            ('category', OneHotEncoder(handle_unknown='ignore'), ['type'])])
        return make_pipeline(prep, estimator)
    return [pipeline(RandomForestClassifier(n_estimators=300, n_jobs=2, random_state=42)),
        pipeline(XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=1, reg_lambda=1,
            objective='binary:logistic', eval_metric='logloss', tree_method='hist',
            n_jobs=2, random_state=42)),
        pipeline(CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05,
            loss_function='Logloss', random_seed=42, thread_count=2,
            verbose=False, allow_writing_files=False))]

sensor_X = sensor_features(train)
sensor_test = sensor_features(test)
y = train.machine_failure

# Compare positive-row weights, keeping features, seeds and threshold fixed.
weight_oof = {w: np.zeros(len(train)) for w in [1, 2, 4]}
weight_folds = []
for fold, (a,b) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(sensor_X,y),1):
    row = {'fold': fold}
    for w in [1,2,4]:
        estimators = sensor_models(base_num + added_num)
        for estimator in estimators:
            if w == 1:
                estimator.fit(sensor_X.iloc[a], y.iloc[a])
            else:
                key = list(estimator.named_steps)[-1] + '__sample_weight'
                estimator.fit(sensor_X.iloc[a], y.iloc[a],
                    **{key: np.where(y.iloc[a] == 1, w, 1.0)})
        prob = np.mean([m.predict_proba(sensor_X.iloc[b])[:,1] for m in estimators],axis=0)
        weight_oof[w][b] = prob
        row[str(w)] = f1_score(y.iloc[b],prob>=0.5)
    weight_folds.append(row)
    print(row,flush=True)
weight_comparison = pd.DataFrame([{'positive_weight':w,
    'F1':f1_score(y,p>=0.5),'precision':precision_score(y,p>=0.5),
    'recall':recall_score(y,p>=0.5)} for w,p in weight_oof.items()]).set_index('positive_weight')
print(weight_comparison.to_string())
# Select by OOF F1, retaining weight 1 if tied. Selection score is not an untouched test score.
best_weight = int(weight_comparison.F1.idxmax())
print('Selected weight:',best_weight)
final_weight_models = sensor_models(base_num+added_num)
for estimator in final_weight_models:
    if best_weight == 1:
        estimator.fit(sensor_X,y)
    else:
        key = list(estimator.named_steps)[-1] + '__sample_weight'
        estimator.fit(sensor_X,y,**{key:np.where(y==1,best_weight,1.0)})
weighted_test_prob = np.mean([m.predict_proba(sensor_test)[:,1] for m in final_weight_models],axis=0)
submission = pd.DataFrame({'id':test.id,'machine_failure':(weighted_test_prob>=0.5).astype(int)})
submission = sample[['id']].merge(submission,on='id',how='left',validate='one_to_one')
assert submission.id.tolist()==sample.id.tolist()
assert submission.machine_failure.isin([0,1]).all()
submission.to_csv('submission.csv',index=False)
print('Submission counts:',submission.machine_failure.value_counts().to_dict())


{'fold': 1, '1': 0.8461538461538461, '2': 0.8333333333333334, '4': 0.8363636363636363}
{'fold': 2, '1': 0.8686868686868687, '2': 0.8686868686868687, '4': 0.8686868686868687}
{'fold': 3, '1': 0.8333333333333334, '2': 0.84, '4': 0.8316831683168316}
{'fold': 4, '1': 0.875, '2': 0.8775510204081632, '4': 0.8712871287128713}
{'fold': 5, '1': 0.8932038834951457, '2': 0.8846153846153846, '4': 0.8518518518518519}
                       F1  precision    recall
positive_weight                               
1                0.863454   0.947137  0.793358
2                0.860511   0.920168  0.808118
4                0.851638   0.891129  0.815498
Selected weight: 1
Submission counts: {0: 1940, 1: 60}


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sample = pd.read_csv('data/sample_submission.csv')
base_num = ['air_temperature_k', 'process_temperature_k',
            'rotational_speed_rpm', 'torque_nm', 'tool_wear_min']
added_num = ['temperature_gap_k', 'mechanical_power_w', 'torque_wear']
def sensor_features(data):
    x = data[base_num + ['type']].copy()
    x['temperature_gap_k'] = x.process_temperature_k - x.air_temperature_k
    x['mechanical_power_w'] = x.torque_nm * x.rotational_speed_rpm * (2 * np.pi / 60)
    x['torque_wear'] = x.torque_nm * x.tool_wear_min
    return x

def sensor_models(numeric):
    def pipeline(estimator):
        prep = ColumnTransformer([
            ('numeric', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
            ('category', OneHotEncoder(handle_unknown='ignore'), ['type'])])
        return make_pipeline(prep, estimator)
    return [pipeline(RandomForestClassifier(n_estimators=300, n_jobs=2, random_state=42)),
        pipeline(XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=1, reg_lambda=1,
            objective='binary:logistic', eval_metric='logloss', tree_method='hist',
            n_jobs=2, random_state=42)),
        pipeline(CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05,
            loss_function='Logloss', random_seed=42, thread_count=2,
            verbose=False, allow_writing_files=False))]

sensor_X = sensor_features(train)
sensor_test = sensor_features(test)
y = train.machine_failure

from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.base import clone

def boundary_features(data):
    x = sensor_features(data)
    for category in ['L','M','H']:
        gate = (x['type'] == category).astype(float)
        x['wear_' + category] = x.tool_wear_min * gate
        x['load_wear_' + category] = x.torque_wear * gate
    return x
boundary_X = boundary_features(train)
boundary_test = boundary_features(test)
boundary_num = base_num + added_num + [c for c in boundary_X if c.startswith(('wear_', 'load_wear_'))]
def boundary_model(kind):
    prep = ColumnTransformer([
        ('numeric',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),boundary_num),
        ('category',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['type'])],sparse_threshold=0)
    if kind == 'GB':
        estimator = GradientBoostingClassifier(n_estimators=400,learning_rate=0.05,
            max_depth=3,min_samples_leaf=3,random_state=42)
    else:
        estimator = HistGradientBoostingClassifier(max_iter=300,learning_rate=0.05,
            max_leaf_nodes=15,min_samples_leaf=10,l2_regularization=1,
            early_stopping=False,random_state=42)
    return make_pipeline(prep,estimator)


# Selected candidate: 50% previous ensemble + 50% specialized Gradient Boosting.
# Five-fold OOF F1: previous 0.863454; combined 0.875502.
# This is a model-selection estimate after error analysis, not an untouched test score.
boundary_final = boundary_model('GB').fit(boundary_X,y)
prior_final = sensor_models(base_num+added_num)
for estimator in prior_final:
    estimator.fit(sensor_X,y)
prior_probability = np.mean([m.predict_proba(sensor_test)[:,1] for m in prior_final],axis=0)
boundary_probability = boundary_final.predict_proba(boundary_test)[:,1]
combined_probability = (prior_probability + boundary_probability) / 2
submission = pd.DataFrame({'id':test.id,'machine_failure':(combined_probability>=0.5).astype(int)})
submission = sample[['id']].merge(submission,on='id',how='left',validate='one_to_one')
assert submission.id.tolist()==sample.id.tolist()
assert submission.machine_failure.isin([0,1]).all()
submission.to_csv('submission.csv',index=False)
print('Boundary blend submission:',submission.shape,submission.machine_failure.value_counts().to_dict())


Boundary blend submission: (2000, 2) {0: 1941, 1: 59}


## 최종 제출 확인
아래 셀은 submission.csv를 직접 만든 뒤 실행하세요.

In [3]:
submission = pd.read_csv("submission.csv")
assert submission.columns.tolist() == sample.columns.tolist()
assert submission.iloc[:, 0].tolist() == sample.iloc[:, 0].tolist()
assert submission.iloc[:, 0].is_unique and not submission.isna().any().any()
print("제출 양식·ID·순서 확인 완료:", submission.shape)


제출 양식·ID·순서 확인 완료: (2000, 2)
